# Сравнение моделей машинного обучения для предсказания цены автомобиля

В этом ноутбуке мы сравним несколько моделей машинного обучения для задачи регрессии — предсказания цены автомобиля на основе характеристик.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from pathlib import Path
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.linear_model import Ridge

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("Warning: XGBoost not installed")

try:
    from catboost import CatBoostRegressor
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print("Warning: CatBoost not installed")

sns.set_theme(style="whitegrid")

## Загрузка данных и подготовка

Загружаем обработанные данные из `data/processed/`, разделяем на признаки и целевую переменную, выполняем train/test split в соотношении 80/20.

In [ ]:
data_path = Path("../data/processed/")
csv_files = list(data_path.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError("No CSV files found in data/processed/")

df = pd.read_csv(csv_files[0])
print(f"Loaded data from: {csv_files[0].name}")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
target_col = "price"
X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## Функция оценки моделей

Функция `evaluate()` вычисляет MAE, RMSE и R² для предсказаний модели.

In [ ]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    print(f"MAE:  {mae:,.2f}")
    print(f"RMSE: {rmse:,.2f}")
    print(f"R²:   {r2:.4f}")
    
    return {"MAE": mae, "RMSE": rmse, "R2": r2}

## Обучение моделей

Обучаем следующие модели:
- DecisionTreeRegressor
- RandomForestRegressor
- GradientBoostingRegressor
- XGBRegressor (если установлен)
- CatBoostRegressor (если установлен)
- Ансамбль (VotingRegressor из лучших 3)

In [ ]:
models = {
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
}

if XGB_AVAILABLE:
    models["XGBoost"] = XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)

if CATBOOST_AVAILABLE:
    models["CatBoost"] = CatBoostRegressor(iterations=100, verbose=0, random_state=42, thread_count=-1)

print(f"Total models to train: {len(models)}")

In [ ]:
results = []
trained_models = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    metrics = evaluate(model, X_test, y_test)
    metrics["Model"] = name
    results.append(metrics)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("MAE").reset_index(drop=True)
print("\n" + "="*50)
print("Сводная таблица результатов (отсортирована по MAE):")
print(results_df[["Model", "MAE", "RMSE", "R2"]].to_string(index=False))

## Ансамбль моделей

Создаём VotingRegressor из трёх лучших моделей по MAE.

In [ ]:
top_3_names = results_df["Model"].head(3).tolist()
top_3_models = [(name, trained_models[name]) for name in top_3_names]

print(f"Creating VotingRegressor from: {top_3_names}")

ensemble = VotingRegressor(estimators=top_3_models)
ensemble.fit(X_train, y_train)

print("\nEnsemble metrics:")
ensemble_metrics = evaluate(ensemble, X_test, y_test)
ensemble_metrics["Model"] = "Ensemble"
results.append(ensemble_metrics)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("MAE").reset_index(drop=True)

## Кросс-валидация для топ-3 моделей

Проводим 5-кратную кросс-валидацию для трёх лучших моделей и выводим mean ± std по MAE.

In [ ]:
top_3_for_cv = results_df.head(3)["Model"].tolist()

print("Кросс-валидация (cv=5) для топ-3 моделей:\n")

cv_results = {}
for name in top_3_for_cv:
    model = trained_models[name]
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
    mae_mean = -scores.mean()
    mae_std = scores.std()
    cv_results[name] = (mae_mean, mae_std)
    print(f"{name}: MAE = {mae_mean:,.2f} ± {mae_std:,.2f}")

## График сравнения моделей

Горизонтальный barplot с MAE каждой модели.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sorted_df = results_df.sort_values("MAE", ascending=True)
ax.barh(sorted_df["Model"], sorted_df["MAE"], color="steelblue")
ax.set_xlabel("MAE")
ax.set_title("Сравнение моделей по MAE")
ax.invert_yaxis()

for i, v in enumerate(sorted_df["MAE"]):
    ax.text(v + 50, i, f"{v:,.0f}", va="center")

plt.tight_layout()

figures_dir = Path("../reports/figures")
figures_dir.mkdir(parents=True, exist_ok=True)
plot_path = figures_dir / "model_comparison_mae.png"
plt.savefig(plot_path, dpi=150)
print(f"График сохранён в: {plot_path}")
plt.show()

## Сохранение лучшей модели

Сохраняем лучшую модель (с наименьшим MAE) в `models/final_model.pkl`.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / "final_model.pkl"

with open(model_path, "wb") as f:
    pickle.dump(best_model, f)

print(f"Лучшая модель ({best_model_name}) сохранена в: {model_path}")

## Вывод

По результатам сравнения моделей:

In [ ]:
print("=" * 60)
print("ИТОГОВЫЙ ВЫВОД")
print("=" * 60)
print(f"\nВыбранная модель: {best_model_name}")
print(f"Причина: Наименьшее значение MAE на тестовой выборке")
print(f"\nМетрики лучшей модели:")
print(f"  MAE:  {results_df.iloc[0]['MAE']:,.2f}")
print(f"  RMSE: {results_df.iloc[0]['RMSE']:,.2f}")
print(f"  R²:   {results_df.iloc[0]['R2']:.4f}")
print("\nТоп-3 модели по MAE:")
for i, row in results_df.head(3).iterrows():
    print(f"  {i+1}. {row['Model']} — MAE: {row['MAE']:,.2f}")